In [0]:
def parse_mail_details(destination_details):
    import json

    def find_to_value(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "to":
                    return v
                result = find_to_value(v)
                if result is not None:
                    return result
        elif isinstance(obj, list):
            for item in obj:
                result = find_to_value(item)
                if result is not None:
                    return result
        return None

    try:
        data = json.loads(destination_details)
        return find_to_value(data)
    except Exception:
        return None

def parse_filesystem_details(destination_details):
    import json

    def find_path_value(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "directory":
                    return v
                result = find_path_value(v)
                if result is not None:
                    return result
        elif isinstance(obj, list):
            for item in obj:
                result = find_path_value(item)
                if result is not None:
                    return result
        return None

    try:
        data = json.loads(destination_details)
        return find_path_value(data)
    except Exception:
        return None

df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details")
completed_df = df.filter(
    (df.schedule_status == "Completed") &
    (df.destination_type.isin("mail","filesystem"))
)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

parse_mail_details_udf = udf(parse_mail_details, StringType())
parse_filesystem_details_udf = udf(parse_filesystem_details, StringType())

completed_df = completed_df.withColumn(
    "sent_to",
    parse_mail_details_udf(completed_df['destination'])
).withColumn(
    "file_path",
    parse_filesystem_details_udf(completed_df['destination'])
)

completed_df.write.mode("overwrite").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched")